In [26]:
import io
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import torch
from transformers import TrainingArguments

from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    DataCollatorWithPadding,
)
from transformers.trainer_utils import EvalPrediction, IntervalStrategy
import math


In [8]:
# ----------------------------
# 2) Load data via pandas
# ----------------------------
# df = pd.read_csv(io.StringIO("train.csv"))
df = pd.read_csv("train.csv")

# Keep only ABSTRACT and the 6 label columns (in the requested order)
LABEL_COLS = [
    "Computer Science",
    "Physics",
    "Mathematics",
    "Statistics",
    "Quantitative Biology",
    "Quantitative Finance",
]

if not set(LABEL_COLS).issubset(df.columns):
    raise ValueError("Expected label columns not found in the CSV data.")

# Clean abstract text column (strip whitespace)
df["text"] = df["ABSTRACT"].astype(str).str.strip()

# Convert labels to 0/1 ints (ensure numeric)
for c in LABEL_COLS:
    df[c] = df[c].astype(int).clip(0, 1)

# For demonstration: If dataset is extremely small (like 1 row),
# duplicate it a few times so training/evaluation can run meaningfully.
# (This preserves the original content while allowing a train/eval split.)
MIN_ROWS_REQUIRED = 4
if len(df) < MIN_ROWS_REQUIRED:
    times = math.ceil(MIN_ROWS_REQUIRED / len(df))
    df = pd.concat([df] * times, ignore_index=True)
    df = df.reset_index(drop=True)

# Build label vectors column
df["labels"] = df[LABEL_COLS].values.tolist()

# Keep only needed columns for datasets
df_for_ds = df[["text", "labels"]].copy()

In [9]:
# ----------------------------
# 3) Train / Eval split (80/20)
# ----------------------------
train_df, eval_df = train_test_split(df_for_ds, test_size=0.2, random_state=42, shuffle=True)

# Convert to Hugging Face datasets
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True))
dataset = DatasetDict({"train": train_ds, "validation": eval_ds})



In [31]:
# ----------------------------
# 4) Tokenizer & Model setup
# ----------------------------
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = len(LABEL_COLS)
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess_function(examples: Dict[str, List[Any]]):
    # Tokenize the texts
    tokenized = tokenizer(
        examples["text"],
        padding=True,  # will be handled by data collator too
        truncation=True,
        max_length=MAX_LENGTH,
    )
    # Keep labels (list of lists) as they are
    tokenized["labels"] = examples["labels"]
    return tokenized

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

# Convert label lists to float tensors expected for multi-label
# (Trainer will collate them into tensors)
# def convert_labels_to_float(example):
#     example["labels"] = [float(x) for x in example["labels"]]
#     return example

# addistional fix for multi-label classification
def convert_labels_to_float(example):
    example["labels"] = [float(x) for x in example["labels"]]
    return example

tokenized_datasets = tokenized_datasets.map(convert_labels_to_float, batched=False)

# additional fix for multi-label classification
tokenized_datasets.set_format(
    "torch", 
    columns=['input_ids', 'attention_mask', 'labels'], 
    output_all_columns=True
)

# additional fix for multi-label classification
# Now, we manually cast the labels column to float for all splits:
for split in tokenized_datasets.keys():
    # This line ensures the labels tensor is explicitly torch.float32
    tokenized_datasets[split] = tokenized_datasets[split].map(
        lambda x: {'labels': torch.tensor(x['labels'], dtype=torch.float32)},
        batched=False
    )

# Data collator (will pad to longest in batch)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load model with problem_type set for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)



Map:   0%|          | 0/16777 [00:00<?, ? examples/s]C:\Users\pc\AppData\Local\Temp\ipykernel_10540\985431484.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lambda x: {'labels': torch.tensor(x['labels'], dtype=torch.float32)},
Map: 100%|██████████| 4195/4195 [00:00<00:00, 5300.24 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
# ----------------------------
# 5) Metrics
# ----------------------------
import numpy as np
import math
from scipy.special import expit  # sigmoid

def compute_metrics(p: EvalPrediction) -> Dict[str, float]:
    """
    p.predictions: logits (shape: batch_size x num_labels)
    p.label_ids: ground-truth labels (shape: batch_size x num_labels)
    We compute:
     - Macro-averaged F1 (multi-label): average the F1 per label
     - Macro-averaged ROC-AUC (multi-label): average roc_auc_score per label
    """
    logits = p.predictions
    labels = p.label_ids

    # For multi-label: apply sigmoid to logits to get probabilities
    probs = expit(logits)  # shape same as logits
    # Binarize predictions using 0.5 threshold for F1
    y_pred = (probs >= 0.5).astype(int)
    y_true = labels.astype(int)

    results: Dict[str, float] = {}

    # Macro F1
    try:
        mac_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    except Exception as e:
        # In rare cases (e.g., degenerate labels), fallback
        mac_f1 = float("nan")

    results["macro_f1"] = float(mac_f1)

    # Macro ROC-AUC: roc_auc_score supports multilabel with shape (#samples, #labels)
    # But it fails if a label has only one class present in y_true.
    # We'll compute roc_auc per label where possible and average those.
    per_label_roc = []
    for i in range(y_true.shape[1]):
        col_true = y_true[:, i]
        col_prob = probs[:, i]
        unique_vals = np.unique(col_true)
        if len(unique_vals) == 1:
            # ROC AUC is undefined when only one class present. Skip this label.
            continue
        try:
            score = roc_auc_score(col_true, col_prob)
            per_label_roc.append(score)
        except Exception:
            # Skip problematic labels
            continue

    if len(per_label_roc) == 0:
        results["macro_roc_auc"] = float("nan")
    else:
        results["macro_roc_auc"] = float(np.mean(per_label_roc))

    return results



In [33]:
# ----------------------------
# 6) TrainingArguments & Trainer
# ----------------------------
output_dir = "./multi_label_demo_output"


training_args = TrainingArguments(
    output_dir=output_dir,
    # evaluation_strategy="epoch",     # OK
    eval_strategy=IntervalStrategy.EPOCH,  # <--- FIX: Renamed from 'evaluation_strategy'
    save_strategy="epoch",           # FIX (no => epoch)
    logging_strategy="epoch",        # OK
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=42,
    load_best_model_at_end=False,
    fp16=torch.cuda.is_available(),   # OK
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


C:\Users\pc\AppData\Local\Temp\ipykernel_10540\3539364196.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [34]:
# ----------------------------
# 7) Train (small demo)
# ----------------------------
print("===== Starting training (demo) =====")
train_result = trainer.train()
print("===== Training complete =====")


===== Starting training (demo) =====


c:\Users\pc\Secondary-Documents\Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


RuntimeError: result type Float can't be cast to the desired output type Long